# Tool Calling in OpenAI and Provider APIs

Orchestration frameworks wrap provider APIs, but production debugging requires reading **raw tool-call payloads**: `tools`, `tool_choice`, `parallel_tool_calls`, and the `tool_calls` block on assistant messages.

This notebook walks through:

- **OpenAI** `chat.completions.create` with `tools` and `tool_choice`
- **Parallel tool calls** — multiple tools in one assistant turn
- A complete **two-turn conversation** loop
- **Anthropic** and **Google Gemini** format differences
- **Raw HTTP** with `httpx` when you need direct API access
- A **provider-agnostic adapter** pattern in plain Python

The scenario is an **Order Support API** (product search, order lookup, policy docs, support tickets). Everything is self-contained — no prior notebooks or frameworks required.

In [ ]:
"""
Order Support API — backend and tool implementations.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


PRODUCT_CATALOG = [
    {"sku": "LAP-001", "name": "Pro Laptop 14", "price": 1299.00, "stock": 12},
    {"sku": "MON-002", "name": "4K Monitor 27", "price": 449.00, "stock": 34},
    {"sku": "KEY-003", "name": "Mechanical Keyboard", "price": 129.00, "stock": 0},
]

ORDERS = {
    "ORD-1001": {
        "customer": "alice@shop.com",
        "status": "shipped",
        "tracking": "1Z999AA10123456784",
    },
}

POLICY_DOCS = [
    {"id": "ship-01", "text": "Standard shipping 5-7 business days. Express 2 days for $15."},
    {"id": "ret-02", "text": "Returns within 30 days if unused. Refunds in 5 business days."},
]


def search_products(query: str) -> list[dict]:
    q = query.lower()
    return [p for p in PRODUCT_CATALOG if q in p["name"].lower() or q in p["sku"].lower()]


def lookup_order(order_id: str) -> dict:
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"found": False, "error": f"Order {order_id} not found"}
    return {"found": True, "order_id": order_id.upper(), **order}


def search_policy(query: str) -> list[dict]:
    q = query.lower()
    return [d for d in POLICY_DOCS if any(t in d["text"].lower() for t in q.split())]


def create_support_ticket(customer_email: str, subject: str, body: str) -> dict:
    return {
        "success": True,
        "ticket_id": f"TKT-{uuid.uuid4().hex[:6].upper()}",
        "customer": customer_email,
        "subject": subject,
    }


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_products": search_products,
    "lookup_order": lookup_order,
    "search_policy": search_policy,
    "create_support_ticket": create_support_ticket,
}

print(f"Tools registered: {list(TOOL_REGISTRY.keys())}")

---

## 1. Provider-Agnostic Tool Call Lifecycle

Every major provider follows the same lifecycle — only field names differ:

```
1. You send:   messages + tool definitions
2. Model returns: assistant message + tool_calls[] (possibly parallel)
3. You execute: tools locally in your runtime
4. You send:   tool result messages linked by call id
5. Model returns: final natural-language answer (or more tool calls)
```

The model **never executes** tools. Your application is always the executor.

---

## 2. OpenAI Tool Definitions

OpenAI expects a `tools` array. Each entry has `type: "function"` and a `function` object with `name`, `description`, and `parameters` (JSON Schema).

This is the exact shape sent in `chat.completions.create(..., tools=OPENAI_TOOLS)`.

In [ ]:
OPENAI_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": (
                "Look up order status and tracking by order ID (e.g. ORD-1001). "
                "Do NOT use for policy questions — use search_policy."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "Order ID from confirmation email.",
                    }
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_policy",
            "description": (
                "Search shipping and returns policy docs. "
                "Do NOT use when you have an order ID — use lookup_order."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Keywords like refund, shipping."}
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search product catalog by name or SKU.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
]

print("lookup_order schema:")
print(pretty(OPENAI_TOOLS[0]))

---

## 3. OpenAI SDK — `chat.completions.create(tools=...)`

The canonical OpenAI call pattern. Set `USE_LIVE_LLM = True` with a valid API key to run against the live API.

In [ ]:
USE_LIVE_LLM = False

MESSAGES = [
    {"role": "user", "content": "Where is order ORD-1001 and what is your return policy?"},
]


def call_openai_once(messages: list[dict], tool_choice: Any = "auto") -> dict:
    """Single OpenAI completion with tools — returns the assistant message dict."""
    from openai import OpenAI

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=OPENAI_TOOLS,
        tool_choice=tool_choice,
        parallel_tool_calls=True,
    )
    msg = response.choices[0].message
    result: dict[str, Any] = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        result["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
            }
            for tc in msg.tool_calls
        ]
    return result


if USE_LIVE_LLM:
    try:
        live_msg = call_openai_once(MESSAGES)
        print(pretty(live_msg))
    except Exception as exc:
        print("Live call failed:", exc)
else:
    print("Offline mode. Request shape:")
    print(f"  model: gpt-4o-mini")
    print(f"  messages: {len(MESSAGES)} user turn(s)")
    print(f"  tools: {[t['function']['name'] for t in OPENAI_TOOLS]}")
    print(f"  tool_choice: auto | parallel_tool_calls: True")

---

## 4. `tool_choice` Modes

Control whether and which tools the model may call on a given turn.

| Value | Behavior |
|-------|----------|
| **`"auto"`** | Model may answer in text or call tools (default) |
| **`"none"`** | Disable tools for this turn |
| **`"required"`** | Model must emit at least one tool call |
| **`{"type":"function","function":{"name":"lookup_order"}}`** | Force a specific tool |

Use `"required"` and forced tools sparingly — they can cause nonsense calls on chitchat like "thanks!".

In [ ]:
TOOL_CHOICE_MODES = {
    "auto": "auto",
    "disable_tools": "none",
    "must_call_tool": "required",
    "force_lookup_order": {"type": "function", "function": {"name": "lookup_order"}},
}

WHEN_TO_USE = {
    "auto": "Default — model decides based on user message.",
    "disable_tools": "User says thanks / follow-up that needs no data.",
    "must_call_tool": "RAG pattern — force retrieval before answering.",
    "force_lookup_order": "User provided order ID — skip tool selection ambiguity.",
}

for label, choice in TOOL_CHOICE_MODES.items():
    print(f"{label}:")
    print(f"  value: {pretty(choice) if isinstance(choice, dict) else repr(choice)}")
    print(f"  when:  {WHEN_TO_USE[label]}")
    print()

---

## 5. `parallel_tool_calls`

When `parallel_tool_calls=True` (OpenAI default on supported models), one assistant turn may include **multiple** `tool_calls` — e.g. look up order **and** search return policy simultaneously.

Rules:

1. Execute **all** tool calls (they are independent unless your domain says otherwise).
2. Append **one tool message per call**, each with the matching `tool_call_id`.
3. Re-prompt the model **once** with all observations.

Set `parallel_tool_calls=False` when downstream systems cannot handle concurrent side effects.

In [ ]:
def execute_openai_tool_call(tool_call: dict) -> dict:
    """Execute one OpenAI-format tool call; return OpenAI-format tool message."""
    name = tool_call["function"]["name"]
    args = json.loads(tool_call["function"]["arguments"])

    if name not in TOOL_REGISTRY:
        content = json.dumps({"error": f"Unknown tool: {name}"})
    else:
        try:
            result = TOOL_REGISTRY[name](**args)
            content = result if isinstance(result, str) else json.dumps(result, default=str)
        except Exception as exc:
            content = json.dumps({"error": str(exc)})

    return {"role": "tool", "tool_call_id": tool_call["id"], "content": content}


def execute_all_tool_calls(tool_calls: list[dict]) -> list[dict]:
    return [execute_openai_tool_call(tc) for tc in tool_calls]


# Simulated parallel assistant message (matches OpenAI SDK response shape)
parallel_assistant = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_order",
            "type": "function",
            "function": {"name": "lookup_order", "arguments": json.dumps({"order_id": "ORD-1001"})},
        },
        {
            "id": "call_policy",
            "type": "function",
            "function": {"name": "search_policy", "arguments": json.dumps({"query": "returns refund"})},
        },
    ],
}

tool_messages = execute_all_tool_calls(parallel_assistant["tool_calls"])
print(f"{len(parallel_assistant['tool_calls'])} parallel calls → {len(tool_messages)} tool messages\n")
for tm in tool_messages:
    print(f"  [{tm['tool_call_id']}] {tm['content'][:75]}...")

---

## 6. Complete Two-Turn Conversation

```
Turn 1: user question  → assistant tool_calls
Turn 2: tool results   → assistant final answer
```

Below: full loop with offline simulated model, then the same structure for live OpenAI.

In [ ]:
def offline_model_response(messages: list[dict]) -> dict:
    """Simulate model: if no tool results yet, emit parallel calls; else final answer."""
    has_tool_results = any(m.get("role") == "tool" for m in messages)
    if not has_tool_results:
        return {
            "role": "assistant",
            "content": None,
            "tool_calls": parallel_assistant["tool_calls"],
        }
    return {
        "role": "assistant",
        "content": (
            "Order ORD-1001 is shipped (tracking 1Z999AA10123456784). "
            "Returns accepted within 30 days if unused [ret-02]."
        ),
    }


def run_openai_tool_loop(
    user_question: str,
    model_fn: Callable[[list[dict]], dict] | None = None,
    max_turns: int = 5,
) -> list[dict]:
    """Provider-agnostic loop using OpenAI message shapes."""
    model_fn = model_fn or offline_model_response
    history: list[dict] = [
        {"role": "system", "content": "You are an order support assistant. Use tools."},
        {"role": "user", "content": user_question},
    ]

    for _ in range(max_turns):
        assistant = model_fn(history)
        history.append(assistant)
        if not assistant.get("tool_calls"):
            break
        history.extend(execute_all_tool_calls(assistant["tool_calls"]))

    return history


history = run_openai_tool_loop("Where is order ORD-1001 and what is the return policy?")
print("Conversation transcript:\n")
for i, msg in enumerate(history):
    role = msg["role"]
    if msg.get("tool_calls"):
        detail = f"tool_calls={[tc['function']['name'] for tc in msg['tool_calls']]}"
    else:
        detail = (msg.get("content") or "")[:70]
    print(f"  [{i}] {role:10} {detail}")

In [ ]:
def run_live_openai_loop(user_question: str, max_turns: int = 5) -> str:
    """Full live loop using OpenAI SDK."""
    history: list[dict] = [
        {"role": "system", "content": "You are an order support assistant."},
        {"role": "user", "content": user_question},
    ]
    for _ in range(max_turns):
        assistant = call_openai_once(history)
        history.append(assistant)
        if not assistant.get("tool_calls"):
            return assistant.get("content") or ""
        history.extend(execute_all_tool_calls(assistant["tool_calls"]))
    return "max_turns reached"


if USE_LIVE_LLM:
    print(run_live_openai_loop("Where is order ORD-1001?"))
else:
    print("Live loop ready — set USE_LIVE_LLM=True to run against OpenAI.")

---

## 7. Anthropic, Google — Format Comparison

The lifecycle is identical; field names and message shapes differ.

| Aspect | OpenAI | Anthropic | Google (Gemini) |
|--------|--------|-----------|-----------------|
| Tool declaration | `tools[].function` | `tools[].input_schema` | `function_declarations` |
| Tool choice | `tool_choice` | `tool_choice` | `tool_config` |
| Parallel calls | `parallel_tool_calls` | supported | supported |
| Result message | `role: "tool"` + `tool_call_id` | `role: "user"` + `tool_result` block | `functionResponse` part |
| Arguments | JSON string in `function.arguments` | object in `input` | object in `args` |

Always check the latest provider documentation — APIs evolve.

In [ ]:
# --- Anthropic tool definition shape ---
ANTHROPIC_TOOLS = [
    {
        "name": "lookup_order",
        "description": "Look up order status by order ID.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]

# Anthropic tool use block in assistant response
ANTHROPIC_ASSISTANT_TOOL_USE = {
    "role": "assistant",
    "content": [
        {
            "type": "tool_use",
            "id": "toolu_01ABC",
            "name": "lookup_order",
            "input": {"order_id": "ORD-1001"},
        }
    ],
}

# Anthropic tool result — note: role is "user", not "tool"
ANTHROPIC_TOOL_RESULT = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": "toolu_01ABC",
            "content": json.dumps(lookup_order("ORD-1001")),
        }
    ],
}

# --- Google Gemini function declaration shape ---
GEMINI_TOOLS = [
    {
        "function_declarations": [
            {
                "name": "lookup_order",
                "description": "Look up order status by order ID.",
                "parameters": {
                    "type": "object",
                    "properties": {"order_id": {"type": "string"}},
                    "required": ["order_id"],
                },
            }
        ]
    }
]

GEMINI_FUNCTION_CALL = {
    "function_call": {"name": "lookup_order", "args": {"order_id": "ORD-1001"}}
}

GEMINI_FUNCTION_RESPONSE = {
    "function_response": {
        "name": "lookup_order",
        "response": lookup_order("ORD-1001"),
    }
}

print("Anthropic tool_use:")
print(pretty(ANTHROPIC_ASSISTANT_TOOL_USE))
print("\nAnthropic tool_result (role=user!):")
print(pretty(ANTHROPIC_TOOL_RESULT))
print("\nGemini function_call:")
print(pretty(GEMINI_FUNCTION_CALL))

---

## 8. Provider Adapter — Normalize Tool Calls

A thin adapter lets your executor work across providers by normalizing tool calls to a common internal shape.

In [ ]:
@dataclass
class NormalizedToolCall:
    id: str
    name: str
    arguments: dict[str, Any]
    provider: str


def normalize_openai_tool_calls(assistant_msg: dict) -> list[NormalizedToolCall]:
    calls = []
    for tc in assistant_msg.get("tool_calls", []):
        calls.append(NormalizedToolCall(
            id=tc["id"],
            name=tc["function"]["name"],
            arguments=json.loads(tc["function"]["arguments"]),
            provider="openai",
        ))
    return calls


def normalize_anthropic_tool_use(assistant_msg: dict) -> list[NormalizedToolCall]:
    calls = []
    for block in assistant_msg.get("content", []):
        if block.get("type") == "tool_use":
            calls.append(NormalizedToolCall(
                id=block["id"],
                name=block["name"],
                arguments=block["input"],
                provider="anthropic",
            ))
    return calls


def execute_normalized(call: NormalizedToolCall) -> Any:
    fn = TOOL_REGISTRY.get(call.name)
    if not fn:
        return {"error": f"unknown tool: {call.name}"}
    return fn(**call.arguments)


def to_openai_tool_message(call: NormalizedToolCall, result: Any) -> dict:
    content = result if isinstance(result, str) else json.dumps(result, default=str)
    return {"role": "tool", "tool_call_id": call.id, "content": content}


def to_anthropic_tool_result(call: NormalizedToolCall, result: Any) -> dict:
    content = result if isinstance(result, str) else json.dumps(result, default=str)
    return {
        "role": "user",
        "content": [{"type": "tool_result", "tool_use_id": call.id, "content": content}],
    }


# Demo: same execution, different response shapes
openai_calls = normalize_openai_tool_calls(parallel_assistant)
anthropic_calls = normalize_anthropic_tool_use(ANTHROPIC_ASSISTANT_TOOL_USE)

for call in openai_calls[:1]:
    result = execute_normalized(call)
    print("OpenAI tool message:", pretty(to_openai_tool_message(call, result)))

for call in anthropic_calls:
    result = execute_normalized(call)
    print("Anthropic tool result:", pretty(to_anthropic_tool_result(call, result)))

---

## 9. Raw HTTP with `httpx`

When SDKs lag new API fields, call the REST endpoint directly. The payload mirrors what the OpenAI Python SDK sends.

In [ ]:
OPENAI_CHAT_URL = "https://api.openai.com/v1/chat/completions"

HTTP_PAYLOAD = {
    "model": "gpt-4o-mini",
    "messages": [
        {"role": "user", "content": "Where is order ORD-1001?"}
    ],
    "tools": OPENAI_TOOLS,
    "tool_choice": "auto",
    "parallel_tool_calls": True,
}

HTTP_HEADERS = {
    "Authorization": f"Bearer {OPENAI_API_KEY}",
    "Content-Type": "application/json",
}


def post_openai_chat(payload: dict) -> dict:
    import httpx
    response = httpx.post(OPENAI_CHAT_URL, headers=HTTP_HEADERS, json=payload, timeout=60.0)
    response.raise_for_status()
    return response.json()


print("HTTP request payload keys:", list(HTTP_PAYLOAD.keys()))
print("tools count:", len(HTTP_PAYLOAD["tools"]))

if USE_LIVE_LLM:
    try:
        data = post_openai_chat(HTTP_PAYLOAD)
        msg = data["choices"][0]["message"]
        print("tool_calls:", msg.get("tool_calls"))
        print("content:", msg.get("content"))
    except Exception as exc:
        print("HTTP call failed:", exc)
else:
    print("\nOffline — payload ready for httpx.post(...)")

---

## 10. Streaming Tool Calls

Streaming completions may emit **partial** tool call JSON across multiple chunks. Your executor must **buffer** until arguments form valid JSON.

```
chunk 1: tool_calls[0].function.name = "lookup_order"
chunk 2: tool_calls[0].function.arguments = "{\"order"
chunk 3: tool_calls[0].function.arguments = "_id\": \"ORD-1001\"}"
         → now safe to parse and execute
```

Never execute a tool call until `json.loads(arguments)` succeeds.

In [ ]:
class StreamingToolCallBuffer:
    """Accumulate streamed argument fragments until JSON is valid."""

    def __init__(self) -> None:
        self.calls: dict[int, dict[str, str]] = {}

    def add_delta(self, index: int, name: str | None = None, arguments_fragment: str | None = None) -> None:
        if index not in self.calls:
            self.calls[index] = {"name": "", "arguments": ""}
        if name:
            self.calls[index]["name"] = name
        if arguments_fragment:
            self.calls[index]["arguments"] += arguments_fragment

    def get_complete_calls(self) -> list[dict]:
        complete = []
        for idx, call in sorted(self.calls.items()):
            try:
                args = json.loads(call["arguments"])
                complete.append({"index": idx, "name": call["name"], "arguments": args})
            except json.JSONDecodeError:
                pass  # still buffering
        return complete


# Simulate streamed fragments
buffer = StreamingToolCallBuffer()
fragments = [
    (0, "lookup_order", None),
    (0, None, '{"order_id": '),
    (0, None, '"ORD-1001"}'),
]

for index, name, frag in fragments:
    buffer.add_delta(index, name, frag)
    complete = buffer.get_complete_calls()
    print(f"After fragment: complete={complete}")

print("\nReady to execute:", buffer.get_complete_calls())

---

## 11. Provider Errors and Debugging

| Error | Cause | Mitigation |
|-------|-------|------------|
| 400 invalid schema | Unsupported JSON Schema feature | Simplify; avoid deep `oneOf` |
| 400 tool_call_id mismatch | Wrong id on tool message | Copy id exactly from response |
| Empty `tool_calls` | `tool_choice="none"` or chitchat | Adjust prompt or force tool |
| Duplicate parallel calls | Model uncertainty | Narrow tool descriptions |
| 429 rate limit | Too many loop iterations | `max_turns`, caching |

**Debugging checklist:**

1. Log request `tools` and response `tool_calls`.
2. Verify `arguments` is valid JSON before executing.
3. Confirm every `tool_call_id` has a matching tool message.
4. Test with `tool_choice` forced per tool.
5. Compare hand-written schema vs generated schema byte-for-byte.

In [ ]:
def validate_openai_transcript(messages: list[dict]) -> tuple[bool, str]:
    pending: set[str] = set()
    for m in messages:
        for tc in m.get("tool_calls", []):
            pending.add(tc["id"])
        if m.get("role") == "tool":
            tid = m.get("tool_call_id")
            if tid not in pending:
                return False, f"orphan tool message: {tid}"
            pending.discard(tid)
    if pending:
        return False, f"missing tool results for: {pending}"
    return True, "valid"


ok, msg = validate_openai_transcript(history)
print(f"Transcript validation: {msg}")

DEBUG_PROMPTS = [
    ("Where is order ORD-1001?", "lookup_order"),
    ("What is your return policy?", "search_policy"),
    ("Do you have laptops in stock?", "search_products"),
    ("Thanks, that helps!", "none — use tool_choice=none"),
]

print("\nDebug prompt → expected tool:")
for prompt, expected in DEBUG_PROMPTS:
    print(f"  {prompt!r} → {expected}")

---

## 12. Security — Treat Tool Arguments as Untrusted

Tool arguments come from the **model**, not directly from the user. They are still **untrusted input**:

- Validate with Pydantic or JSON Schema before execution.
- Never pass tool args directly to SQL, shell, or file paths.
- Scope credentials per tool — read-only DB role for search tools.
- Log every tool call for audit.

The API contract is secure by design (model cannot execute code). Your **executor** must enforce boundaries.

In [ ]:
BLOCKED_ORDER_IDS = {"ORD-ADMIN", "ORD-DELETE-ALL"}


def lookup_order_safe(order_id: str) -> dict:
    """Executor-level guardrail — block suspicious model-generated ids."""
    oid = order_id.upper().strip()
    if oid in BLOCKED_ORDER_IDS:
        return {"error": "access denied", "order_id": oid}
    if not re.match(r"^ORD-\d+$", oid):
        return {"error": "invalid order_id format", "received": order_id}
    return lookup_order(oid)


MALICIOUS_CALL = {
    "id": "call_evil",
    "type": "function",
    "function": {"name": "lookup_order", "arguments": json.dumps({"order_id": "ORD-DELETE-ALL"})},
}

LEGIT_CALL = {
    "id": "call_ok",
    "type": "function",
    "function": {"name": "lookup_order", "arguments": json.dumps({"order_id": "ORD-1001"})},
}

for label, tc in [("malicious", MALICIOUS_CALL), ("legitimate", LEGIT_CALL)]:
    args = json.loads(tc["function"]["arguments"])
    result = lookup_order_safe(args["order_id"])
    print(f"{label}: {result}")

---

## 13. Check Your Understanding

1. What four fields does an OpenAI `tool_calls` entry contain?
2. What is the difference between `tool_choice: "auto"` and `"required"`?
3. When the model emits two parallel tool calls, how many tool messages do you send back?
4. How does Anthropic's tool result message differ from OpenAI's?
5. Why must you buffer streamed tool call arguments before executing?
6. Why are tool arguments considered untrusted even though the user did not type them?
7. What does a provider adapter normalize across OpenAI and Anthropic?

---

## 14. Summary

- **`chat.completions.create(..., tools=)`** is the OpenAI entry point; **`tool_choice`** and **`parallel_tool_calls`** control invocation shape.
- Execute **every** `tool_call`; reply with **`tool_call_id`-linked** tool messages before re-prompting.
- **Anthropic** uses `tool_use` / `tool_result` blocks; **Gemini** uses `function_call` / `function_response` — same lifecycle, different JSON.
- A **provider adapter** normalizes tool calls so your executor stays provider-agnostic.
- **Streamed** tool calls require buffering until arguments are valid JSON.
- **Validate and sandbox** tool arguments — they come from the model and are untrusted.

Understanding raw provider payloads makes framework abstractions debuggable. When tool calling fails in production, inspect the request `tools` array and response `tool_calls` first.